In [1]:
import sys, wrds
print(sys.version)
print("wrds:", wrds.__version__)


3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
wrds: 3.4.0


In [2]:
import numpy as np
import pandas as pd
import wrds

# ============================================================
# dolvol (monthly dollar volume, with t-2 lag)
# Output: dolvol_raw = ln(|PRC| * VOL) at month t-2, labeled at month t (calendar month-end)
# ============================================================

# -----------------------
# 0) Parameters
# -----------------------
OUT_START = "1957-01-31"
OUT_END   = "1989-12-31"

# Because dolvol_t uses t-2, we need inputs starting at OUT_START - 2 months
IN_START  = "1956-11-01"

OUTFILE   = "dolvol_raw_1957_1989_all_months.csv"

# -----------------------
# 1) Connect WRDS
# -----------------------
db = wrds.Connection()

# -----------------------
# 2) Pull CRSP monthly data + common share filter (shrcd 10/11)
#    Use msenames to get the correct shrcd for each date (namedt <= date <= nameendt)
# -----------------------
query = f"""
select
    a.permno, a.date, a.prc, a.vol,
    b.shrcd
from crsp.msf as a
left join crsp.msenames as b
    on a.permno = b.permno
    and b.namedt <= a.date
    and a.date  <= b.nameendt
where a.date >= '{IN_START}'
  and a.date <= '{OUT_END}'
"""
msf = db.raw_sql(query, date_cols=["date"])

# Keep only common shares
msf = msf[msf["shrcd"].isin([10, 11])].copy()

# -----------------------
# 3) Compute dollar volume on input month: DollarVol = |PRC| * VOL
#    Then take log: ln(DollarVol)
# -----------------------
prc = pd.to_numeric(msf["prc"], errors="coerce").to_numpy(dtype=float)
vol = pd.to_numeric(msf["vol"], errors="coerce").to_numpy(dtype=float)

dollar_vol = np.abs(prc) * vol
dollar_vol[dollar_vol <= 0] = np.nan  # avoid log(0) or negative

msf["dvol_input"] = np.log(dollar_vol)

# -----------------------
# 4) Map input month -> output month t = input + 2 months
#    Use calendar month-end labels
# -----------------------
msf["in_month"] = msf["date"].dt.to_period("M")
msf["date_out"] = (msf["in_month"] + 2).dt.to_timestamp("M")  # calendar month-end

# -----------------------
# 5) Build output panel (permno, date, dolvol_raw), within OUT_START..OUT_END
# -----------------------
dolvol = (
    msf.loc[
        (msf["date_out"] >= pd.Timestamp(OUT_START)) &
        (msf["date_out"] <= pd.Timestamp(OUT_END)),
        ["permno", "date_out", "dvol_input"]
    ]
    .rename(columns={"date_out": "date", "dvol_input": "dolvol_raw"})
    .drop_duplicates(["permno", "date"])
    .sort_values(["date", "permno"])
    .reset_index(drop=True)
)

# -----------------------
# 6) Save + quick checks
# -----------------------
dolvol.to_csv(OUTFILE, index=False)

print(dolvol.head())
print("min date:", dolvol["date"].min(), "| max date:", dolvol["date"].max())
print("unique months:", dolvol["date"].nunique())
print("unique permno:", dolvol["permno"].nunique())
print("saved:", OUTFILE)


Enter your WRDS username [zhouzixian]: zixian_zhou
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  n


You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
   permno       date  dolvol_raw
0   10006 1957-01-31    9.569953
1   10014 1957-01-31    6.237836
2   10022 1957-01-31    7.008844
3   10030 1957-01-31    9.825337
4   10057 1957-01-31    7.901007
min date: 1957-01-31 00:00:00 | max date: 1989-12-31 00:00:00
unique months: 396
unique permno: 13473
saved: dolvol_raw_1957_1989_all_months.csv
